**Extración de la data y bibliotecas**
--

Se importa todas las librerías necesarias para realizar cada operación de este notebook

Recuerda ejectuar ```pip install requirements.txt``` para instalar las librerías.

In [1]:
import pandas as pd

Se extrae del archivo "Movies.csv" la información de las peliculas y el registro de las valores.
Luego, se muestra como se ve la data.

In [2]:
data = pd.read_csv(filepath_or_buffer='./data/Movies.csv', delimiter=',', encoding='utf-8')
data.head(100)

,index,revenue,vote_average,vote_count,title,original_language,release_date,production_companies,production_countries,genres,director,producer,cast,runtimes,writer
0,0,4300000,6.6,714,Four Rooms,en,1995-12-09,"Miramax Films,A Band Apart",United States of America,"Crime,Comedy","Allison Anders,Alexandre Rockwell,Robert Rodri...","Lawrence Bender,Paul Hellerman,Scott Lambert,","Sammi Davis,Amanda De Cadenet,Valeria Golino,",98.0,"Allison Anders,Alexandre Rockwell,Robert Rodri..."
1,1,12,6.4,96,Judgment Night,en,1993-10-15,"Universal Pictures,Largo Entertainment,JVC Ent...","Japan,United States of America","Action,Thriller,Crime",Stephen Hopkins,"Gene Levy,Christopher Mollo,Lloyd Segan,","Emilio Estevez,Cuba Gooding Jr.,Denis Leary,",110.0,"Lewis Colick,Jere Cunningham,Lewis Colick"
2,2,775398007,8.1,8550,Star Wars,en,1977-05-25,"Lucasfilm,20th Century Fox",United States of America,"Adventure,Action,Science Fiction",George Lucas,"Gary Kurtz,George Lucas,Rick McCallum","Mark Hamill,Harrison Ford,Carrie Fisher,",121.0,George Lucas
3,3,940335536,7.7,8085,Finding Nemo,en,2003-05-30,Pixar,United States of America,"Animation,Family","Andrew Stanton,Lee Unkrich","Jinko Gotoh,John Lasseter,Graham Walters","Albert Brooks,Ellen DeGeneres,Alexander Gould,",100.0,"Andrew Stanton,Andrew Stanton,Bob Peterson,"
4,4,677945399,8.3,10346,Forrest Gump,en,1994-07-06,Paramount,United States of America,"Comedy,Drama,Romance",Robert Zemeckis,"Wendy Finerman,Charles Newirth,Steve Starkey,","Tom Hanks,Rebecca Williams,Sally Field,",142.0,"Winston Groom,Eric Roth"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,158733820,7.3,3439,Sin City,en,2005-04-01,"Miramax,Dimension Films,Troublemaker Studios",United States of America,"Action,Thriller,Crime","Frank Miller,Robert Rodriguez,Quentin Tarantino","Elizabeth Avellan,Bill Scott,Bob Weinstein,","Jessica Alba,Devon Aoki,Alexis Bledel,",124.0,Frank Miller
96,96,39407616,6.3,1637,Sin City: A Dame to Kill For,en,2014-08-20,"Residaco,Miramax Films,Troublemaker Studios,Al...",United States of America,"Crime,Thriller","Frank Miller,Robert Rodriguez","Elizabeth Avellan,Marina Bespalov,Sergei Bespa...","Mickey Rourke,Jessica Alba,Josh Brolin,",102.0,"Frank Miller,Frank Miller"
97,97,120000000,6.4,545,Star Trek: Generations,en,1994-11-17,Paramount,United States of America,"Science Fiction,Action,Adventure,Thriller",David Carson,"Rick Berman,Peter Lauritson,Bernard Williams","Patrick Stewart,Jonathan Frakes,Brent Spiner,",118.0,"Gene Roddenberry,Rick Berman,Ronald D. Moore,"
98,98,0,7.4,65,Trouble in Paradise,en,1932-10-21,Paramount,United States of America,"Comedy,Romance",Ernst Lubitsch,Ernst Lubitsch,"Miriam Hopkins,Kay Francis,Herbert Marshall,",83.0,"Samson Raphaelson,Grover Jones,Aladar Laszlo"


**Limpieza de Datos**
--

Aqui se elimina todas las filas que no son utiles para el analisís, debido que esta sujeto a las siguientes condiciones:

- La película no tiene:
    - Una compañia de producción cinematográfico
    - Fecha de lanzamiento
    - Director
    - Productor
    - Actores
    - Genero
    - Escritor
- La duración de la película sea igual a 0
- La cantidad de votos de una película es igual a 0
- El promedio de la valoración de la película es igual a 0

In [3]:
# Elimina las películas que tenga valores nulos o falta información en uno o más columnas.

data = data.dropna(subset=[
    'production_companies', 
    'release_date',
    'runtimes',
    'genres',
    'director',
    'producer',
    'cast',
    'writer'
])

data = data[data['vote_count'] > 0]
data = data[data['vote_average'] > 0]

print(len(data))


8044


Dado que el dataset se compone mayormente de películas en ingles como idioma, no sera relevante para el análisis.
Por ende, se elimina la columna *original_language* del dataset. 

In [4]:
data = data.drop('original_language', axis=1)

**Transformación de Datos**
--

En las siguientes columnas, se identificado que una parte del dataset su contenido finaliza con una coma (","):

- Columna *cast*: información de los actores (principales) que participan en una película.
- Columna *producer*: información de los productor(es) de una película.

Se revisa por lineas, para eliminar el caracter adicional ubicado al final del texto.

In [5]:
for i, row in data.iterrows():
    if row['cast'][-1:] == ',':
        data.loc[i, 'cast'] = row['cast'][:-1]

    if row['producer'][-1:] == ',':
        data.loc[i, 'producer'] = row['producer'][:-1]

**Análisis de Datos**
--

El siguiente análisis, se quiere obtener las estadísticas sobre los generos de peliculas:
- Cantidad de veces que se menciona un genero del dataset.
- Que porcentaje representa un genero dentro del dataset.

Ha de considerar que una película puede ser categorizado por más de 1 genero. 
El siguiente fragmento, obtengo todos los generos únicos identificados del dataset.

In [6]:
# Una película puede tener más de un genero, separo los valores y lo almaceno en un arreglo
generos = data['genres'].str.split(',')

# Se identifica los valores unicos
generos = generos.explode().unique()
generos


array(['Crime', 'Comedy', 'Action', 'Thriller', 'Adventure',
       'Science Fiction', 'Animation', 'Family', 'Drama', 'Romance',
       'Mystery', 'Music', 'Documentary', 'Fantasy', 'War', 'Western',
       'History', 'Horror', 'TV Movie'], dtype=object)

In [7]:
genre_stat = []
total = len(data)
for genre in generos:

    count = int(data['genres'].str.contains(genre, na=False).sum())
    percent = round((count / total) * 100, 2)
    genre_stat.append([genre, count, percent])

genre_stat

[['Crime', 1288, 16.01],
 ['Comedy', 2913, 36.21],
 ['Action', 1770, 22.0],
 ['Thriller', 2067, 25.7],
 ['Adventure', 1123, 13.96],
 ['Science Fiction', 916, 11.39],
 ['Animation', 417, 5.18],
 ['Family', 817, 10.16],
 ['Drama', 3477, 43.22],
 ['Romance', 1409, 17.52],
 ['Mystery', 676, 8.4],
 ['Music', 354, 4.4],
 ['Documentary', 207, 2.57],
 ['Fantasy', 651, 8.09],
 ['War', 234, 2.91],
 ['Western', 255, 3.17],
 ['History', 232, 2.88],
 ['Horror', 1220, 15.17],
 ['TV Movie', 134, 1.67]]

Repetimos el mismo procedimiento para el caso del pais que se producío una película. Igualmente, esta la situación en donde una película es producido en más de 1 pais.


In [8]:
# Una película puede tener más de un pais en el que se producío, separo los valores y lo almaceno en un arreglo
paises = data['production_countries'].str.split(',')

# Se identifica los valores unicos
paises = paises.explode().unique()
paises

paises_stat = []
total = len(data)

for pais in paises:

    count = int(data['production_countries'].str.contains(pais, na=False).sum())
    percent = round((count / total) * 100, 4)
    paises_stat.append([pais, count, percent])

paises_stat

[['United States of America', 8044, 100.0],
 ['Japan', 99, 1.2307],
 ['Argentina', 4, 0.0497],
 ['Denmark', 10, 0.1243],
 ['Finland', 3, 0.0373],
 ['France', 203, 2.5236],
 ['Germany', 338, 4.2019],
 ['Iceland', 7, 0.087],
 ['Italy', 99, 1.2307],
 ['Netherlands', 20, 0.2486],
 ['Norway', 3, 0.0373],
 ['Sweden', 7, 0.087],
 ['United Kingdom', 565, 7.0239],
 ['Canada', 278, 3.456],
 ['Austria', 7, 0.087],
 ['Switzerland', 7, 0.087],
 ['Hong Kong', 24, 0.2984],
 ['Uruguay', 1, 0.0124],
 ['Paraguay', 1, 0.0124],
 ['Australia', 78, 0.9697],
 ['New Zealand', 18, 0.2238],
 ['China', 11, 0.1367],
 ['Taiwan', 4, 0.0497],
 ['South Africa', 19, 0.2362],
 ['Hungary', 9, 0.1119],
 ['Spain', 48, 0.5967],
 ['Czech Republic', 16, 0.1989],
 ['Ecuador', 1, 0.0124],
 ['Colombia', 1, 0.0124],
 ['India', 17, 0.2113],
 ['Malta', 3, 0.0373],
 ['Ireland', 28, 0.3481],
 ['Bahamas', 2, 0.0249],
 ['Lebanon', 1, 0.0124],
 ['Mexico', 35, 0.4351],
 ['Bulgaria', 9, 0.1119],
 ['Costa Rica', 1, 0.0124],
 ['Morocco', 4

El siguiente fragmento, se obtiene las estadísticas de los actores identificados en el casting de cada película.

In [9]:
# Una película puede tener más de un pais en el que se producío, separo los valores y lo almaceno en un arreglo
actores = data['cast'].str.split(',')

# Se identifica los valores unicos
actores = actores.explode().unique()
actores

actores_stat = []
total = len(data)

for actor in actores:

    count = int(data['cast'].str.contains(actor, na=False).sum())
    percent = round((count / total) * 100, 4)
    actores_stat.append([actor, count, percent])

actores_stat

[['Sammi Davis', 1, 0.0124],
 ['Amanda De Cadenet', 1, 0.0124],
 ['Valeria Golino', 5, 0.0622],
 ['Emilio Estevez', 16, 0.1989],
 ['Cuba Gooding Jr.', 18, 0.2238],
 ['Denis Leary', 11, 0.1367],
 ['Mark Hamill', 11, 0.1367],
 ['Harrison Ford', 24, 0.2984],
 ['Carrie Fisher', 7, 0.087],
 ['Albert Brooks', 7, 0.087],
 ['Ellen DeGeneres', 1, 0.0124],
 ['Alexander Gould', 1, 0.0124],
 ['Tom Hanks', 31, 0.3854],
 ['Rebecca Williams', 1, 0.0124],
 ['Sally Field', 21, 0.2611],
 ['Kevin Spacey', 15, 0.1865],
 ['Annette Bening', 14, 0.174],
 ['Thora Birch', 7, 0.087],
 ['Joseph Cotten', 19, 0.2362],
 ['Dorothy Comingore', 1, 0.0124],
 ['Agnes Moorehead', 3, 0.0373],
 ['Björk', 2, 0.0249],
 ['Catherine Deneuve', 4, 0.0497],
 ['David Morse', 12, 0.1492],
 ['Michael Hynson', 1, 0.0124],
 ['Robert August', 2, 0.0249],
 ['Lord James Blears', 1, 0.0124],
 ['Johnny Depp', 26, 0.3232],
 ['Geoffrey Rush', 12, 0.1492],
 ['Orlando Bloom', 7, 0.087],
 ['Uma Thurman', 16, 0.1989],
 ['Lucy Liu', 12, 0.1492],
